# Teacher SFT + Targeted DPO 实验（E14）

**前置条件**：E11-E13 已完成，以下文件已在 Google Drive 上：
- `outputs/group_teacher_sft/merged` — Teacher SFT 模型（E12 产出）
- `logs/eval_supplement_teacher_sft_gsm8k.json` — E13 GSM8K 评测结果
- `logs/eval_supplement_teacher_sft_math.json` — E13 MATH-500 评测结果
- `logs 2/eval_supplement_teacher_sft_gsm8k_badcases.jsonl` — E13 badcase
- `data/processed/sft_teacher_gsm8k.json` — E11 处理后的 teacher SFT 数据
- `data/processed/dpo_targeted_by_type_v1/` — v1 targeted DPO 数据

**本 notebook 内容**：
1. 环境设置 + Drive 挂载
2. 读取 E13 结果
3. E14: 合并 v1 + badcase → DPO 训练 → 评测

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# E0: 环境准备 + Drive 挂载
# ═══════════════════════════════════════════════════════════════════
!pip install -q -U pip
!pip install -q "unsloth>=2025.1.0" "trl>=0.14.0" "peft>=0.14.0" "bitsandbytes>=0.45.0" \
    "transformers>=4.49.0" "datasets>=3.2.0" "accelerate>=1.2.0" \
    "pyyaml>=6.0.2" "safetensors" "tqdm" "scipy" "sympy" "requests"

import torch, os, json, subprocess, sys
from pathlib import Path
print(f'PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name} | VRAM: {gpu.total_memory / 1e9:.1f} GB')

EVAL_N = '200'
BIT = ['--load_in_4bit']

def run_eval(cmd, label):
    cmd = [cmd[0], '-u'] + cmd[1:]
    print(f'  执行: {" ".join(cmd[-8:])}')
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    if proc.returncode != 0:
        print(f'  ❌ {label} 失败 (exit {proc.returncode})')
        return False
    return True

def print_result(path, label):
    if not os.path.isfile(path):
        print(f'  {label}: 结果文件不存在 ({path})')
        return
    d = json.load(open(path))
    acc = d.get('accuracy', d.get('macro_avg_accuracy', 'N/A'))
    total = d.get('total', 'N/A')
    if isinstance(acc, float):
        print(f'  {label}: {acc:.1%} ({total}题)')
    else:
        print(f'  {label}: {acc} (n={total})')

def is_eval_complete(output, expected=None):
    if not os.path.isfile(output):
        return False
    try:
        d = json.load(open(output))
        details = d.get('details', [])
        if expected is not None and len(details) < int(expected):
            print(f'  ⏩ {output}: 已有 {len(details)}/{expected} 条，续跑...')
            return False
        return True
    except Exception:
        return False

# 挂载 Drive
from google.colab import drive, userdata
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/Qwen-Reasoning'
os.chdir(PROJECT_DIR)
print(f'工作目录: {PROJECT_DIR}')

# 拉取最新代码
if os.path.isdir(f'{PROJECT_DIR}/.git'):
    subprocess.run(['git', 'pull', '--rebase'], check=False)

# 设置 API key
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
    print('HF_TOKEN: OK')
except Exception:
    print('HF_TOKEN: missing')

sys.path.insert(0, f'{PROJECT_DIR}/eval')
sys.path.insert(0, f'{PROJECT_DIR}/scripts')
for d in ['logs', 'outputs']:
    os.makedirs(d, exist_ok=True)

print('✅ 环境就绪')

In [ ]:
# ── 读取 E13 结果 ─────────────────────────────────────────────
os.chdir(PROJECT_DIR)

print('\n' + '='*60)
print('  E13 结果读取')
print('='*60)

# Teacher SFT 模型
G_TS_MERGED = 'outputs/group_teacher_sft/merged'
if os.path.isfile(f'{G_TS_MERGED}/config.json'):
    print(f'✅ Teacher SFT 模型: {G_TS_MERGED}')
else:
    print(f'❌ Teacher SFT 模型不存在: {G_TS_MERGED}')

# E13 评测结果
print_result('logs/eval_supplement_teacher_sft_gsm8k.json', 'Teacher SFT GSM8K')
print_result('logs/eval_supplement_teacher_sft_math.json', 'Teacher SFT MATH-500')

# 检查 E14 前置文件
print('\n📋 E14 前置文件检查:')
for label, path in [
    ('v1 targeted DPO 数据', 'data/processed/dpo_targeted_by_type_v1'),
    ('Teacher SFT badcases', 'logs 2/eval_supplement_teacher_sft_gsm8k_badcases.jsonl'),
    ('Teacher SFT 数据', 'data/processed/sft_teacher_gsm8k.json'),
]:
    if os.path.exists(path):
        if os.path.isdir(path):
            n = len([f for f in os.listdir(path) if f.endswith('.json')])
            print(f'  ✅ {label}: {path} ({n} json files)')
        else:
            print(f'  ✅ {label}: {path}')
    else:
        print(f'  ❌ {label}: {path} 不存在')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# E14: Teacher SFT + Targeted DPO（v1 + Teacher badcase 合并）
# 策略：v1 targeted DPO (426条) + Teacher SFT badcase (70条)
#       合并后对 chosen 做精简处理，训练 DPO，再评测
# Base: outputs/group_teacher_sft/merged（Teacher SFT 基座）
# ═══════════════════════════════════════════════════════════════════
os.chdir(PROJECT_DIR)

import re, subprocess

print('\n' + '='*60)
print('  E14: Teacher SFT + Targeted DPO')
print('='*60)

# ── 路径定义 ──
G_TT_DPO = 'outputs/group_teacher_targeted/dpo'
G_TT_MERGED = 'outputs/group_teacher_targeted/merged'
TEACHER_SFT_BASE = 'outputs/group_teacher_sft/merged'
TEACHER_DPO_DATA = 'data/processed/dpo_teacher_targeted.json'
V1_DIR = 'data/processed/dpo_targeted_by_type_v1'
BADCASE_FILE = 'logs 2/eval_supplement_teacher_sft_gsm8k_badcases.jsonl'
SFT_TEACHER_FILE = 'data/processed/sft_teacher_gsm8k.json'

MAX_CHOSEN_LEN = 4096

# ── 文本处理函数 ──
def strip_think_tags(text):
    return re.sub(r'<think>.*?</think>\s*', '', text, flags=re.DOTALL)

def strip_filler(text):
    patterns = [
        r"^(?:Okay|Alright|Well|So|Now),?\s+(?:let's (?:see|try|figure out)|I need to|let me)\b.*?\.\s*",
        r"^Let me (?:think about|work through|solve|figure out)\b.*?\.\s*",
        r"^(?:嗯|好的|首先[，,])\s*",
    ]
    for p in patterns:
        text = re.sub(p, '', text, count=1, flags=re.IGNORECASE | re.DOTALL)
    return text.strip()

def strip_correction_paragraphs(text):
    paragraphs = re.split(r'\n\s*\n', text)
    kept = []
    for para in paragraphs:
        corrections = len(re.findall(r'(?:Wait|Actually|Let me re|I made a mistake|Hmm|不对|重新)', para, re.IGNORECASE))
        if corrections < 2:
            kept.append(para)
    return '\n\n'.join(kept).strip()

def truncate_preserve_answer(text, max_len=MAX_CHOSEN_LEN):
    if len(text) <= max_len:
        return text
    boxed = list(re.finditer(r'\\boxed\{[^}]+\}', text))
    hash_ans = list(re.finditer(r'####\s*.+', text))
    answer_pat = boxed[-1] if boxed else (hash_ans[-1] if hash_ans else None)
    if answer_pat:
        end = answer_pat.end()
        if end <= max_len:
            return text[:max_len]
        prefix_len = max_len - (end - answer_pat.start()) - 20
        if prefix_len > 200:
            return text[:prefix_len] + '\n\n[...]\n\n' + text[answer_pat.start():end]
    return text[:max_len]

def process_chosen(text):
    text = strip_think_tags(text)
    text = strip_filler(text)
    text = strip_correction_paragraphs(text)
    text = truncate_preserve_answer(text)
    return text

# ── NF4 检测函数（独立，不依赖 0.3 cell）──
def check_nf4(merged_path):
    """检测 merged 模型是否含 NF4 权重。"""
    if not os.path.isfile(f'{merged_path}/config.json'):
        return None  # 模型不存在
    try:
        from model_loader import _has_quantized_weights
        return _has_quantized_weights(merged_path)
    except ImportError:
        # fallback: 直接检查 safetensors
        import torch as _torch
        from safetensors.torch import load_file
        import glob
        st_files = glob.glob(f'{merged_path}/*.safetensors')
        if st_files:
            tensors = load_file(st_files[0])
            sample = list(tensors.values())[0]
            is_nf4 = sample.dtype in (_torch.uint8,)
            del tensors
            return is_nf4
    return False

def ensure_fp16(merged_path, adapter_path, label):
    """确保 merged 模型是 fp16（非 NF4），否则重合并。"""
    if not os.path.isfile(f'{merged_path}/config.json'):
        return merged_path
    is_nf4 = check_nf4(merged_path)
    if not is_nf4:
        return merged_path
    # 需要重合并
    fp16_path = merged_path.rstrip('/') + '_fp16'
    if os.path.isfile(f'{fp16_path}/config.json') and not check_nf4(fp16_path):
        return fp16_path
    print(f'  {label}: NF4 → 重合并 fp16')
    # 读取 base_model
    cfg_path = f'{adapter_path}/adapter_config.json'
    base_model = ''
    if os.path.isfile(cfg_path):
        with open(cfg_path) as f:
            base_model = json.load(f).get('base_model_name_or_path', '')
    cmd = ['python3', 'scripts/merge_lora.py',
           '--adapter_path', adapter_path, '--output_path', fp16_path]
    if base_model:
        cmd += ['--base_model', base_model]
    subprocess.run(cmd, check=True)
    return fp16_path

# ── 断点续跑检测 ──
if os.path.isfile(TEACHER_DPO_DATA):
    existing = json.load(open(TEACHER_DPO_DATA))
    print(f'✅ Targeted DPO 数据已存在: {TEACHER_DPO_DATA} ({len(existing)} 条)')
else:
    print(f'\n── Step 1: 加载 v1 Targeted DPO 数据 ──')
    v1_all = []
    for fname in sorted(os.listdir(V1_DIR)):
        if fname.endswith('.json'):
            fpath = os.path.join(V1_DIR, fname)
            items = json.load(open(fpath))
            v1_all.extend(items)
            print(f'  {fname}: {len(items)} 条')
    print(f'  v1 合计: {len(v1_all)} 条')

    print(f'\n── Step 2: 加载 Teacher SFT badcases ──')
    badcases = []
    with open(BADCASE_FILE) as f:
        for line in f:
            if line.strip():
                badcases.append(json.loads(line))
    print(f'  badcases: {len(badcases)} 条')

    # 加载 SFT teacher 数据以获取 E11 处理后的 output
    sft_teacher = json.load(open(SFT_TEACHER_FILE))
    sft_map = {d['instruction']: d['output'] for d in sft_teacher}

    print(f'\n── Step 3: 构建 badcase DPO 对 ──')
    badcase_pairs = []
    for bc in badcases:
        q = bc['question']
        gt_raw = bc.get('gt_raw', '')
        pred_raw = bc.get('pred_raw', '')
        chosen = sft_map.get(q, process_chosen(gt_raw))
        if not chosen or len(chosen) < 50:
            chosen = process_chosen(gt_raw)
        rejected = pred_raw if pred_raw else str(bc.get('pred', ''))
        if chosen and rejected and chosen != rejected:
            badcase_pairs.append({
                'prompt': q,
                'chosen': process_chosen(chosen),
                'rejected': rejected[:MAX_CHOSEN_LEN],
                'error_type': 'badcase_mixed',
                'source': 'teacher_sft_badcase',
            })
    print(f'  badcase DPO 对: {len(badcase_pairs)} 条')

    print(f'\n── Step 4: 处理 v1 chosen ──')
    v1_pairs = []
    for d in v1_all:
        chosen = process_chosen(d['chosen'])
        rejected = d['rejected']
        if chosen and rejected and len(chosen) > 50 and chosen != rejected:
            v1_pairs.append({
                'prompt': d['prompt'],
                'chosen': chosen,
                'rejected': rejected[:MAX_CHOSEN_LEN],
                'error_type': d.get('error_type', 'unknown'),
                'source': 'v1_targeted',
            })
    print(f'  v1 有效对: {len(v1_pairs)} 条')

    print(f'\n── Step 5: 合并并保存 ──')
    all_pairs = v1_pairs + badcase_pairs
    with open(TEACHER_DPO_DATA, 'w') as f:
        json.dump(all_pairs, f, ensure_ascii=False, indent=2)
    print(f'  合计: {len(all_pairs)} 条 → {TEACHER_DPO_DATA}')

    from collections import Counter
    types = Counter(d['error_type'] for d in all_pairs)
    print(f'  错误类型分布: {dict(types)}')
    chosen_lens = [len(d['chosen']) for d in all_pairs]
    print(f'  chosen 长度: avg={sum(chosen_lens)//len(chosen_lens)}, max={max(chosen_lens)}')

# ── DPO 训练 ──
print(f'\n── Step 6: DPO 训练 ──')
if not os.path.isfile(TEACHER_SFT_BASE + '/config.json'):
    print(f'❌ Teacher SFT base 不存在: {TEACHER_SFT_BASE}')
elif os.path.isfile(f'{G_TT_MERGED}/config.json'):
    print(f'✅ Teacher Targeted DPO merged 已存在: {G_TT_MERGED}')
elif os.path.isfile(f'{G_TT_DPO}/adapter_config.json'):
    print(f'✅ Teacher Targeted DPO adapter 已存在: {G_TT_DPO}')
else:
    dpo_data = json.load(open(TEACHER_DPO_DATA))
    n = len(dpo_data)
    max_steps = max(100, 6 * n // 16)
    warmup_steps = max(10, max_steps // 10)

    print(f'  数据: {n} 条')
    print(f'  训练: {max_steps} steps, warmup={warmup_steps}, lr=1e-5, beta=0.1')
    print(f'  Base: {TEACHER_SFT_BASE}')

    import yaml
    tt_config_path = 'config/dpo_teacher_targeted.yaml'
    tt_cfg = {
        'model_name': 'Qwen/Qwen2.5-1.5B-Instruct',
        'base_adapter_path': TEACHER_SFT_BASE,
        'output_dir': G_TT_DPO,
        'max_seq_length': 2048,
        'load_in_4bit': True,
        'seed': 42,
        'beta': 0.1,
        'loss_type': 'sigmoid',
        'dataset_path': TEACHER_DPO_DATA,
        'lora': {
            'use_dora': False,
            'r': 16,
            'alpha': 32,
            'dropout': 0.05,
            'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                               'gate_proj', 'up_proj', 'down_proj'],
        },
        'train': {
            'per_device_train_batch_size': 1,
            'gradient_accumulation_steps': 16,
            'warmup_steps': warmup_steps,
            'max_steps': max_steps,
            'learning_rate': 1e-5,
            'logging_steps': 10,
            'save_steps': 50,
            'weight_decay': 0.0,
            'lr_scheduler_type': 'cosine',
            'optim': 'paged_adamw_8bit',
            'fp16': False,
            'bf16': True,
            'dataloader_num_workers': 4,
            'dataloader_pin_memory': True,
        },
    }
    with open(tt_config_path, 'w') as f:
        yaml.dump(tt_cfg, f, default_flow_style=False, allow_unicode=True)
    print(f'  配置已写入: {tt_config_path}')

    run_eval([
        'python3', '-u', 'scripts/dpo_train.py',
        '--config', tt_config_path,
    ], 'Teacher Targeted DPO 训练')

# ── 合并 LoRA ──
print(f'\n── Step 7: 合并 LoRA → merged ──')
if os.path.isfile(f'{G_TT_DPO}/adapter_config.json') and not os.path.isfile(f'{G_TT_MERGED}/config.json'):
    merge_base = ensure_fp16(TEACHER_SFT_BASE, 'outputs/group_teacher_sft/sft', 'Teacher SFT')
    print(f'  base model: {merge_base}')
    os.makedirs(G_TT_MERGED, exist_ok=True)
    r = subprocess.run([
        'python3', 'scripts/merge_lora.py',
        '--adapter_path', G_TT_DPO,
        '--base_model', merge_base,
        '--output_path', G_TT_MERGED,
    ], capture_output=True, text=True)
    if r.returncode != 0:
        print(f'  ❌ merge_lora.py 失败 (exit {r.returncode})')
        print(f'  stdout: {r.stdout[-500:]}')
        print(f'  stderr: {r.stderr[-500:]}')
    else:
        print(f'  ✅ 合并完成: {G_TT_MERGED}')
elif os.path.isfile(f'{G_TT_MERGED}/config.json'):
    print(f'  ✅ merged 已存在: {G_TT_MERGED}')

# NF4 检测
if os.path.isfile(f'{G_TT_MERGED}/config.json'):
    G_TT_MERGED = ensure_fp16(G_TT_MERGED, G_TT_DPO, 'Teacher Targeted DPO')

# ── 评测 ──
print(f'\n── Step 8: 评测 GSM8K + MATH-500 ──')
if not os.path.isfile(f'{G_TT_MERGED}/config.json'):
    print(f'❌ merged 模型不存在，跳过评测')
else:
    for bench, script in [('GSM8K', 'eval/gsm8k_eval.py'), ('MATH-500', 'eval/math_eval.py')]:
        if bench == 'GSM8K':
            out_file = 'logs/eval_supplement_teacher_targeted_gsm8k.json'
        else:
            out_file = 'logs/eval_supplement_teacher_targeted_math.json'
        if os.path.isfile(out_file):
            r = json.load(open(out_file))
            print(f'  ✅ {bench}: {r.get("accuracy", 0)*100:.1f}% ({r.get("correct")}/{r.get("total")}) — 已存在')
        else:
            print(f'  运行 {bench} 评测...')
            run_eval([
                'python3', '-u', script,
                '--model_path', G_TT_MERGED,
                '--max_samples', EVAL_N,
                '--output', out_file,
            ] + BIT, f'{bench} 评测')
            if os.path.isfile(out_file):
                r = json.load(open(out_file))
                print(f'  ✅ {bench}: {r.get("accuracy", 0)*100:.1f}% ({r.get("correct")}/{r.get("total")})')

print('\nE14 完成')